# Retrieval output generation

# Workflow

Set `MODE = "dev"` while developing and tuning. After the best alpha is selected from the dev split, set `MODE = "test"` and set `RERANK_ALPHA` to the selected alpha.

The retrieval logic is:

BM25 uses `raw_text`, cleans footnote markers, tokenizes with underthesea, and returns top-20.

Dense retrieval encodes the claim with `AITeamVN/Vietnamese_Embedding` and queries Qdrant. The Qdrant vectors are assumed to have been created from `embed_input`.

Hybrid retrieval merges BM25 top-20 and dense top-20 by `chunk_id`, normalizes scores inside the merged candidate set, and computes:

`hybrid_score = alpha * dense_norm + (1 - alpha) * bm25_norm`

The reranker reranks only the hybrid candidates of `RERANK_ALPHA` and saves top-10 and top-5 files.

# Install packages

Run this cell on Colab if the packages are not installed.

In [1]:
!pip -q install rank-bm25 underthesea qdrant-client
!pip -q install "transformers==4.44.2" "tokenizers==0.19.1" "sentence-transformers==3.0.1" "accelerate" "FlagEmbedding==1.2.11"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 14.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggi

# Configuration

In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import re
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from google.colab import userdata
from underthesea import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker
from qdrant_client import QdrantClient

MODE = "dev"  # "dev" or "test"

DEV_PATH = "/content/dev_dataset.jsonl"
TEST_PATH = "/content/test_dataset.jsonl"
CORPUS_PATH = "/content/full_DVSK_data_aiteamvn_256.json"

OUTPUT_ROOT = Path("/content/retrieval_outputs")
OUTPUT_DIR = OUTPUT_ROOT / MODE

QDRANT_COLLECTION = "dvsktt_aiteamvn_chunks_256"
DENSE_MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

CLAIM_ID_FIELD = "claim_id"
CLAIM_FIELD = "claim"
GOLD_EVIDENCE_FIELD = "evidence_ids"
CORPUS_TEXT_FIELD = "raw_text"

FIRST_STAGE_TOP_K = 20
ALPHAS = [0.3, 0.5, 0.7]

# On dev, this can be 0.5 for the first run.
# After evaluation selects the best alpha, set this value to that alpha and rerun retrieval.
# On test, this must be the alpha selected from the dev split.
RERANK_ALPHA = 0.5

RERANK_INPUT_TOP_K = 20
RERANK_OUTPUT_TOP_K = 10
DENSE_BATCH_SIZE = 32
RERANK_BATCH_SIZE = 16

RUN_ALL_ALPHAS_ON_TEST = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "dev":
    DATASET_PATH = DEV_PATH
    EFFECTIVE_ALPHAS = ALPHAS
else:
    DATASET_PATH = TEST_PATH
    EFFECTIVE_ALPHAS = ALPHAS if RUN_ALL_ALPHAS_ON_TEST else [RERANK_ALPHA]

def alpha_tag(alpha: float) -> str:
    return f"alpha{int(round(alpha * 10)):02d}"

print(f"Mode: {MODE}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Hybrid alphas to save: {EFFECTIVE_ALPHAS}")
print(f"Rerank alpha: {RERANK_ALPHA}")

Mode: dev
Dataset path: /content/dev_dataset.jsonl
Output directory: /content/retrieval_outputs/dev
Hybrid alphas to save: [0.3, 0.5, 0.7]
Rerank alpha: 0.5


# Load dataset and corpus

In [4]:
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def read_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Corpus file must be a JSON list of chunk objects.")

    return data

dataset_rows = read_jsonl(DATASET_PATH)
corpus_rows = read_json_list(CORPUS_PATH)

for row in dataset_rows:
    row["split"] = MODE

corpus_by_id = {str(row["chunk_id"]): row for row in corpus_rows}
chunk_ids = list(corpus_by_id.keys())

print(f"Dataset rows: {len(dataset_rows)}")
print(f"Corpus chunks: {len(corpus_rows)}")

Dataset rows: 264
Corpus chunks: 5343


# Connect to Qdrant

In [5]:
qdrant_url = userdata.get("QDRANT_URL")
qdrant_api_key = userdata.get("QDRANT_API_KEY")

qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
    timeout=120,
)

qdrant_count = qdrant_client.count(
    collection_name=QDRANT_COLLECTION,
    exact=True,
).count

print(f"Qdrant collection: {QDRANT_COLLECTION}")
print(f"Qdrant points: {qdrant_count}")

Qdrant collection: dvsktt_aiteamvn_chunks_256
Qdrant points: 5343


# Build BM25 index from raw text

In [6]:
def clean_text_for_bm25(text: str) -> str:
    text = str(text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

bm25_corpus_tokens = []

for cid in tqdm(chunk_ids, desc="Tokenizing corpus for BM25"):
    raw_text = corpus_by_id[cid][CORPUS_TEXT_FIELD]
    cleaned_text = clean_text_for_bm25(raw_text)
    tokens = word_tokenize(cleaned_text, format="text").split()
    bm25_corpus_tokens.append(tokens)

bm25 = BM25Okapi(bm25_corpus_tokens)

print("BM25 index is ready.")

Tokenizing corpus for BM25:   0%|          | 0/5343 [00:00<?, ?it/s]

BM25 index is ready.


# Load dense model and reranker

In [7]:
dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    trust_remote_code=True,
)

reranker = FlagReranker(
    RERANKER_MODEL_NAME,
    use_fp16=torch.cuda.is_available(),
)

print(f"Dense model: {DENSE_MODEL_NAME}")
print(f"Reranker model: {RERANKER_MODEL_NAME}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Dense model: AITeamVN/Vietnamese_Embedding
Reranker model: BAAI/bge-reranker-v2-m3


# Retrieval helpers

In [8]:
def make_evidence_item(cid, rank, score, extra_scores=None):
    source = corpus_by_id[str(cid)]

    item = {
        "rank": rank,
        "chunk_id": str(cid),
        "score": float(score),
        "book_name": source["book_name"],
        "pages": source["pages"],
        "raw_text": source["raw_text"],
    }

    if extra_scores:
        for key, value in extra_scores.items():
            item[key] = float(value)

    return item

def build_output_row(row, method_name, evidence_items):
    output = dict(row)
    output["retrieval_method"] = method_name
    output["retrieved_evidence"] = evidence_items
    return output

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def minmax_normalize(score_map):
    values = np.array(list(score_map.values()), dtype=np.float32)
    min_value = float(values.min())
    max_value = float(values.max())
    if max_value - min_value < 1e-12:
        return {key: 0.0 for key in score_map}
    return {
        key: float((value - min_value) / (max_value - min_value))
        for key, value in score_map.items()
    }

def query_qdrant(vector, top_k):
    result = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=vector.tolist(),
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    points = result.points if hasattr(result, "points") else result
    hits = []

    for point in points:
        payload = point.payload or {}
        cid = str(point.id)
        hits.append((str(cid), float(point.score)))

    return hits

# Run BM25 and dense retrieval

In [9]:
claims = [row[CLAIM_FIELD] for row in dataset_rows]

bm25_hits_by_claim_id = {}
dense_hits_by_claim_id = {}

query_vectors = dense_model.encode(
    claims,
    batch_size=DENSE_BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
)

for row, query_vector in tqdm(list(zip(dataset_rows, query_vectors)), desc="Running BM25 and dense retrieval"):
    claim_id = str(row[CLAIM_ID_FIELD])
    claim = row[CLAIM_FIELD]

    query_tokens = word_tokenize(clean_text_for_bm25(claim), format="text").split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:FIRST_STAGE_TOP_K]
    bm25_hits = [
        (chunk_ids[idx], float(bm25_scores[idx]))
        for idx in bm25_top_indices
    ]

    dense_hits = query_qdrant(query_vector, FIRST_STAGE_TOP_K)

    bm25_hits_by_claim_id[claim_id] = bm25_hits
    dense_hits_by_claim_id[claim_id] = dense_hits

print("BM25 and dense retrieval are finished.")

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Running BM25 and dense retrieval:   0%|          | 0/264 [00:00<?, ?it/s]

BM25 and dense retrieval are finished.


# Save BM25 and dense outputs

In [10]:
method_outputs = {}

for method_name, hits_by_claim_id in [
    ("bm25", bm25_hits_by_claim_id),
    ("dense", dense_hits_by_claim_id),
]:
    top20_rows = []
    top5_rows = []

    for row in dataset_rows:
        claim_id = str(row[CLAIM_ID_FIELD])
        hits = hits_by_claim_id[claim_id]

        top20_evidence = [
            make_evidence_item(cid, rank=index + 1, score=score)
            for index, (cid, score) in enumerate(hits[:20])
        ]
        top5_evidence = top20_evidence[:5]

        top20_rows.append(build_output_row(row, f"{method_name}_top20", top20_evidence))
        top5_rows.append(build_output_row(row, f"{method_name}_top5", top5_evidence))

    method_outputs[f"{method_name}_top20"] = top20_rows
    method_outputs[f"{method_name}_top5"] = top5_rows

    write_jsonl(OUTPUT_DIR / f"{method_name}_top20_all_labels_{MODE}.jsonl", top20_rows)
    write_jsonl(OUTPUT_DIR / f"{method_name}_top5_all_labels_{MODE}.jsonl", top5_rows)

print("BM25 and dense output files are saved.")

BM25 and dense output files are saved.


# Save hybrid outputs

In [11]:
hybrid_hits_by_alpha = {}

for alpha in EFFECTIVE_ALPHAS:
    tag = alpha_tag(alpha)
    hybrid_hits_by_claim_id = {}
    top20_rows = []
    top5_rows = []

    for row in tqdm(dataset_rows, desc=f"Building hybrid retrieval {tag}"):
        claim_id = str(row[CLAIM_ID_FIELD])
        bm25_hits = dict(bm25_hits_by_claim_id[claim_id])
        dense_hits = dict(dense_hits_by_claim_id[claim_id])

        merged_ids = sorted(set(bm25_hits) | set(dense_hits))

        bm25_raw = {cid: float(bm25_hits.get(cid, 0.0)) for cid in merged_ids}
        dense_raw = {cid: float(dense_hits.get(cid, 0.0)) for cid in merged_ids}

        bm25_norm = minmax_normalize(bm25_raw)
        dense_norm = minmax_normalize(dense_raw)

        scored = []
        for cid in merged_ids:
            hybrid_score = alpha * dense_norm[cid] + (1.0 - alpha) * bm25_norm[cid]
            scored.append({
                "chunk_id": cid,
                "score": float(hybrid_score),
                "bm25_score": bm25_raw[cid],
                "dense_score": dense_raw[cid],
                "bm25_norm": bm25_norm[cid],
                "dense_norm": dense_norm[cid],
            })

        scored = sorted(scored, key=lambda item: item["score"], reverse=True)[:FIRST_STAGE_TOP_K]
        hybrid_hits_by_claim_id[claim_id] = scored

        top20_evidence = [
            make_evidence_item(
                item["chunk_id"],
                rank=index + 1,
                score=item["score"],
                extra_scores={
                    "bm25_score": item["bm25_score"],
                    "dense_score": item["dense_score"],
                    "bm25_norm": item["bm25_norm"],
                    "dense_norm": item["dense_norm"],
                },
            )
            for index, item in enumerate(scored)
        ]

        top5_evidence = top20_evidence[:5]

        top20_rows.append(build_output_row(row, f"hybrid_{tag}_top20", top20_evidence))
        top5_rows.append(build_output_row(row, f"hybrid_{tag}_top5", top5_evidence))

    hybrid_hits_by_alpha[tag] = hybrid_hits_by_claim_id
    method_outputs[f"hybrid_{tag}_top20"] = top20_rows
    method_outputs[f"hybrid_{tag}_top5"] = top5_rows

    write_jsonl(OUTPUT_DIR / f"hybrid_{tag}_top20_all_labels_{MODE}.jsonl", top20_rows)
    write_jsonl(OUTPUT_DIR / f"hybrid_{tag}_top5_all_labels_{MODE}.jsonl", top5_rows)

print("Hybrid output files are saved.")

Building hybrid retrieval alpha03:   0%|          | 0/264 [00:00<?, ?it/s]

Building hybrid retrieval alpha05:   0%|          | 0/264 [00:00<?, ?it/s]

Building hybrid retrieval alpha07:   0%|          | 0/264 [00:00<?, ?it/s]

Hybrid output files are saved.


# Save reranker outputs

In [12]:
rerank_tag = alpha_tag(RERANK_ALPHA)

if rerank_tag not in hybrid_hits_by_alpha:
    raise ValueError(
        f"RERANK_ALPHA={RERANK_ALPHA} was not generated in this run. "
        f"Generated alpha tags: {list(hybrid_hits_by_alpha.keys())}"
    )

reranker_top10_rows = []
reranker_top5_rows = []

for row in tqdm(dataset_rows, desc=f"Reranking hybrid candidates {rerank_tag}"):
    claim_id = str(row[CLAIM_ID_FIELD])
    claim = row[CLAIM_FIELD]
    candidates = hybrid_hits_by_alpha[rerank_tag][claim_id][:RERANK_INPUT_TOP_K]

    pairs = []
    for item in candidates:
        raw_text = corpus_by_id[item["chunk_id"]][CORPUS_TEXT_FIELD]
        pairs.append([claim, clean_text_for_bm25(raw_text)])

    rerank_scores = reranker.compute_score(pairs)

    if isinstance(rerank_scores, float):
        rerank_scores = [rerank_scores]

    reranked = []
    for item, rerank_score in zip(candidates, rerank_scores):
        reranked.append({
            **item,
            "reranker_score": float(rerank_score),
        })

    reranked = sorted(
        reranked,
        key=lambda item: item["reranker_score"],
        reverse=True,
    )[:RERANK_OUTPUT_TOP_K]

    top10_evidence = [
        make_evidence_item(
            item["chunk_id"],
            rank=index + 1,
            score=item["reranker_score"],
            extra_scores={
                "hybrid_score": item["score"],
                "bm25_score": item["bm25_score"],
                "dense_score": item["dense_score"],
                "bm25_norm": item["bm25_norm"],
                "dense_norm": item["dense_norm"],
            },
        )
        for index, item in enumerate(reranked)
    ]

    top5_evidence = top10_evidence[:5]

    reranker_top10_rows.append(build_output_row(row, f"hybrid_reranker_{rerank_tag}_top10", top10_evidence))
    reranker_top5_rows.append(build_output_row(row, f"hybrid_reranker_{rerank_tag}_top5", top5_evidence))

method_outputs[f"hybrid_reranker_{rerank_tag}_top10"] = reranker_top10_rows
method_outputs[f"hybrid_reranker_{rerank_tag}_top5"] = reranker_top5_rows

write_jsonl(OUTPUT_DIR / f"hybrid_reranker_{rerank_tag}_top10_all_labels_{MODE}.jsonl", reranker_top10_rows)
write_jsonl(OUTPUT_DIR / f"hybrid_reranker_{rerank_tag}_top5_all_labels_{MODE}.jsonl", reranker_top5_rows)

print("Reranker output files are saved.")

Reranking hybrid candidates alpha05:   0%|          | 0/264 [00:00<?, ?it/s]

Reranker output files are saved.


# Save sanity summary and configuration

In [13]:
summary_rows = []

for method_name, rows in method_outputs.items():
    retrieved_lengths = [len(row["retrieved_evidence"]) for row in rows]
    label_counts = pd.Series([row["label"] for row in rows]).value_counts().to_dict()

    summary_row = {
        "mode": MODE,
        "method": method_name,
        "rows": len(rows),
        "min_retrieved": int(min(retrieved_lengths)) if retrieved_lengths else 0,
        "max_retrieved": int(max(retrieved_lengths)) if retrieved_lengths else 0,
        "mean_retrieved": float(np.mean(retrieved_lengths)) if retrieved_lengths else 0.0,
    }

    for label, count in label_counts.items():
        summary_row[f"label_{label}"] = int(count)

    summary_rows.append(summary_row)

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"retrieval_outputs_sanity_summary_{MODE}.csv"
summary_df.to_csv(summary_path, index=False)

config = {
    "mode": MODE,
    "dataset_path": DATASET_PATH,
    "corpus_path": CORPUS_PATH,
    "output_dir": str(OUTPUT_DIR),
    "qdrant_collection": QDRANT_COLLECTION,
    "dense_model_name": DENSE_MODEL_NAME,
    "reranker_model_name": RERANKER_MODEL_NAME,
    "first_stage_top_k": FIRST_STAGE_TOP_K,
    "effective_alphas": EFFECTIVE_ALPHAS,
    "rerank_alpha": RERANK_ALPHA,
    "rerank_output_top_k": RERANK_OUTPUT_TOP_K,
    "bm25_text_source": "raw_text cleaned and tokenized with underthesea",
    "dense_text_source": "Qdrant vectors precomputed from embed_input",
    "hybrid_formula": "alpha * dense_norm + (1 - alpha) * bm25_norm",
    "hybrid_candidate_set": "BM25 top-20 union Dense top-20",
}

with open(OUTPUT_DIR / f"retrieval_config_{MODE}.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

display(summary_df)

print(f"Saved summary: {summary_path}")
print(f"All outputs are in: {OUTPUT_DIR}")

,mode,method,rows,min_retrieved,max_retrieved,mean_retrieved,label_SUPPORTED,label_REFUTED,label_NOT_ENOUGH_EVIDENCE
0,dev,bm25_top20,264,20,20,20.0,88,88,88
1,dev,bm25_top5,264,5,5,5.0,88,88,88
2,dev,dense_top20,264,20,20,20.0,88,88,88
3,dev,dense_top5,264,5,5,5.0,88,88,88
4,dev,hybrid_alpha03_top20,264,20,20,20.0,88,88,88
5,dev,hybrid_alpha03_top5,264,5,5,5.0,88,88,88
6,dev,hybrid_alpha05_top20,264,20,20,20.0,88,88,88
7,dev,hybrid_alpha05_top5,264,5,5,5.0,88,88,88
8,dev,hybrid_alpha07_top20,264,20,20,20.0,88,88,88
9,dev,hybrid_alpha07_top5,264,5,5,5.0,88,88,88


Saved summary: /content/retrieval_outputs/dev/retrieval_outputs_sanity_summary_dev.csv
All outputs are in: /content/retrieval_outputs/dev


# Cleanup

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()